In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Reproducibility and Inspection

Suppose we change one line of a training script and the validation accuracy
of a Fashion-MNIST classifier rises by 0.3 points. Before we credit the
change, we need to know how much two runs of the unchanged script differ.
For softmax regression we measure the answer in
that section: five runs that differ only in their seed end
one to four points apart, because each run draws different initial weights
and visits the examples in a different order. A gain of 0.3 points is well
inside that range.

A run that fails poses a second problem of the same kind. If the loss turns
into NaN after hours of training, we want to repeat the run exactly up to
that point and then look inside the model to see which layer produced the
first non-finite number.

This section first makes runs repeatable. Every random draw comes from a
generator whose sequence of numbers is fixed by a seed, so seeding the
generators fixes the draws. That is the easy part. Repeating a run bit for
bit also requires that the arithmetic adds numbers in a fixed order, which
a GPU does not promise. The section then looks inside running models with
*hooks*, functions that observe each layer of an unmodified model as it
computes. The two halves serve one purpose: a measurement taken inside a
model can be compared across two versions of the code only if the runs
differ in nothing but that code. On a first reading, the subsections on
generator objects and on loader workers can be skimmed; they matter once a
data pipeline draws random numbers of its own.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [ ]:
from torch.utils.data import DataLoader, Dataset

## Seeds and Random Number Generators

Randomness enters a training run in more places than one might list at
first: initialization draws every weight (that section), dropout
draws a new mask at every step, the data loader shuffles the examples
differently in every epoch, data augmentation draws crops and flips, and
parallel loader workers do some of this work in separate processes. Each
draw comes from a *pseudorandom number generator*, a deterministic algorithm
whose sequence of outputs is fixed by its starting state, the *seed*.
Seeding every generator that a run consults therefore fixes every draw.

### Global Seeds

A framework's random functions either consult a global generator or take
their state as an argument. Which of the two a framework does decides how a
run is seeded. To see a seed at work, the function below trains a small
network for five steps on random data and draws everything, weights and
data, from the one source that its seed fixes.

`torch.manual_seed(k)` seeds PyTorch's global generator on the CPU and on
every GPU. It does not seed Python's `random` module or NumPy's generators,
which carry seeds of their own.

In [ ]:
def train_once(seed):
    torch.manual_seed(seed)  # seeds the CPU and every GPU
    net = nn.Sequential(nn.Linear(20, 32), nn.ReLU(), nn.Linear(32, 1))
    opt = torch.optim.SGD(net.parameters(), lr=0.1)
    X, y = torch.randn(128, 20), torch.randn(128, 1)
    for _ in range(5):
        opt.zero_grad()
        loss = ((net(X) - y) ** 2).mean()
        loss.backward()
        opt.step()
    return loss.item(), net[0].weight.detach().clone()

loss_a, w_a = train_once(seed=0)
loss_b, w_b = train_once(seed=0)
loss_c, _ = train_once(seed=1)
print('same seed, identical loss and weights:',
      loss_a == loss_b, torch.equal(w_a, w_b))
print('different seed:', loss_a, 'vs', loss_c)

The two runs with seed 0 agree *bitwise*, down to the last bit of the loss
and of every weight: same initial weights, same data, same gradients, same
operations in the same order. (The order is the one thing a seed does not
control; we return to it in that section.) The
run with seed 1 ends with a different loss.


### Generator Objects

A single global generator couples all the code that draws from it. Inserting
one extra random call, a new layer's initialization or a stray draw while
debugging, shifts every number drawn after it, so the data split, the
initialization, and the dropout masks all change at once. A *generator
object* is a private stream with its own seed, which unrelated draws cannot
shift.

A `torch.Generator` is such a stream. Here a random permutation, say a split
of a dataset into training and validation examples, stays fixed no matter
what else draws from the global generator in between:

In [ ]:
g = torch.Generator().manual_seed(42)
split = torch.randperm(10, generator=g)
_ = torch.randn(1000)  # unrelated draws from the global generator
split_again = torch.randperm(10, generator=g.manual_seed(42))
print(torch.equal(split, split_again), split)

Most sampling functions accept `generator=`, and so does `DataLoader`, where
it sets the shuffle order. We use that next.

### Data Loader Workers

Data loaders often prepare minibatches in parallel *worker* processes. A
worker that draws random numbers, for data augmentation for instance, needs
a stream of its own. Two failures are common, and neither raises an error.
If every worker starts as a copy of the main process, as with the *fork*
start method, the workers inherit the same generator state and apply the
same "random" augmentations. If each worker seeds itself from the operating
system's entropy, no two runs agree.

`DataLoader` gives every worker its own PyTorch seed, derived from a base
seed and the worker's index, and seeds Python's `random` module and NumPy's
global generator in each worker from it. It cannot reach a generator object
that the dataset holds: every worker receives a copy of the dataset, the
generator's state included. Such generators need a *worker initializer*, a
function that the loader calls once in each new worker;
`get_worker_info()` gives it the worker's seed and its copy of the dataset.
The loader's `generator` sets the base seed (and the shuffle order), so one
generator fixes the whole input pipeline. Below, two workers read a dataset
of eight examples, each a draw from the dataset's own generator, first
without and then with an initializer. The cell asks for the fork start
method, which Linux and macOS provide and Windows does not:

In [ ]:
class NoisyData(Dataset):
    """Eight examples whose values are random draws, as in augmentation."""
    def __init__(self):
        self.rng = np.random.default_rng()  # the dataset's own generator

    def __len__(self):
        return 8

    def __getitem__(self, i):
        return self.rng.random()

def seed_worker(worker_id):
    info = torch.utils.data.get_worker_info()
    info.dataset.rng = np.random.default_rng(info.seed)  # this worker's seed

for init_fn in (None, seed_worker):
    loader = DataLoader(NoisyData(), batch_size=4, num_workers=2,
                        worker_init_fn=init_fn,
                        multiprocessing_context='fork',  # copies of this one
                        generator=torch.Generator().manual_seed(0))
    print([[round(x, 3) for x in batch.tolist()] for batch in loader])

Without the initializer, the two batches hold the same four numbers: both
workers drew from copies of one generator. With it, the numbers differ, and
since the worker seeds derive from the loader's generator, the second line
is the same in every run. The first line changes from run to run, because
the dataset's generator seeded itself from the operating system.

### Randomness as a Value

The worker failures above arise because the state of a generator is copied
or shared without appearing in the program's data flow. JAX makes randomness
a value instead: every random function takes a key, the same key gives the
same numbers, and independent streams come from splitting a key. A key can
still be reused by mistake, but the reuse is visible in the code. The cost
is that every function that draws random numbers must accept a key and pass
split keys on. A global seed is the convenient form of the same contract: one
implicit key that all code shares. Generator objects recover the explicit
form where it matters.

PyTorch offers both forms: the global generator, and `torch.Generator`
objects passed to the functions that should use them.

### The Spread across Seeds

We can now measure the spread that the opening asked about. The function
below trains the softmax regression of that section on
Fashion-MNIST for three epochs on a GPU, seeds every generator that the run
consults, and returns the final validation accuracy and the trained
weights. We run it with the seeds 0 to 4.

`torch.manual_seed` fixes the initialization, the loader's shuffle order,
and the seeds of its worker processes.

In [ ]:
def train_classifier(seed, data):
    torch.manual_seed(seed)  # initialization, shuffle order, worker seeds
    model = d2l.SoftmaxRegression(num_outputs=10, lr=0.1)
    trainer = d2l.Trainer(max_epochs=3, num_gpus=1)
    trainer.board.display = False
    trainer.fit(model, data)
    return trainer.history['val_acc'][-1], model.net[1].weight.detach().cpu()

data = d2l.FashionMNIST(batch_size=256)
runs = [train_classifier(seed, data) for seed in range(5)]
accs = 100 * np.array([acc for acc, _ in runs])
print('validation accuracy (%):', ' '.join(f'{a:.1f}' for a in accs))
print(f'mean {accs.mean():.1f}, max - min {np.ptp(accs):.1f} points')

The five runs differ only in their seeds, yet their final accuracies lie
one to four points apart, around 81%. A gain of 0.3 points, measured on one
run of each variant, is well inside that spread. To credit a change we
compare the mean over several seeds per variant, together with the spread.

The seed is all that separated these runs. To confirm it, we repeat the
run with seed 0 and compare it with the first one, accuracy and every
weight:

In [ ]:
acc, w = train_classifier(0, data)
print('same accuracy:', acc == runs[0][0],
      '| same weights:', torch.equal(w, runs[0][1]),
      f'| largest difference {(w - runs[0][1]).abs().max().item():.2g}')

The rerun reproduces the accuracy of the first run exactly. Whether every
weight agrees as well depends on the order in which the GPU adds numbers,
which a seed does not fix (that section).

Here every weight agrees bitwise: the operations of this model add in a
fixed order on the GPU.

## Deterministic Arithmetic

Seeding fixes which numbers the program draws. It does not fix how the
arithmetic combines them. Floating-point addition is not associative: every
sum is rounded to the nearest representable number, so adding the same
numbers in a different order can give a different result.

Three float32 numbers suffice to show it. Near $10^8$, adjacent float32
numbers are 8 apart (that section), so $-10^8 + 1$ rounds back to
$-10^8$:

In [ ]:
a, b, c = [torch.tensor(v, dtype=torch.float32) for v in (1e8, -1e8, 1.0)]
print(float((a + b) + c), float(a + (b + c)))  # same numbers, regrouped

Regrouping changes the result from 1 to 0. In a long sum the effect is
smaller but of the same kind: adding a million random numbers forward and
backward usually gives results that differ in their last bits.

On a CPU, the order in which one operation adds its numbers is usually fixed
for a given number of threads, so a seeded run repeats. On a GPU, many
threads add partial sums at the same time, and some operations combine them
with *atomic additions*, which commit each partial sum in whatever order the
threads happen to finish. Examples are the gradients of an embedding lookup
or of a scatter operation, some convolution gradients, and some reductions.
Two seeded runs on the same machine can then differ in the last bits of
their weights, and over many training steps such differences grow until the
loss curves differ visibly. The remedy is a switch that replaces such
operations with deterministic ones, which add in a fixed order and are often
slower. The cell below shows both. A *scatter-add* sums a million random
numbers into ten bins, each number into the bin its index names; we run it
twice on a GPU and compare the two results bit by bit.

`torch.use_deterministic_algorithms(True)` is the switch. Operations that
have a deterministic implementation use it; operations without one raise a
`RuntimeError` instead of returning nondeterministic results. The cell runs
the scatter-add `index_add_` with the switch off and then on:

In [ ]:
if torch.cuda.is_available():
    src = torch.randn(2**20, device='cuda')  # a million numbers
    idx = torch.randint(0, 10, (2**20,), device='cuda')  # their bins
    for deterministic in (False, True):
        torch.use_deterministic_algorithms(deterministic)
        a, b = [torch.zeros(10, device='cuda').index_add_(0, idx, src)
                for _ in range(2)]
        print(f'deterministic {deterministic!s:5}: bitwise equal '
              f'{torch.equal(a, b)}, largest difference '
              f'{(a - b).abs().max().item():.2g}')
    torch.use_deterministic_algorithms(False)
else:
    print('no GPU: on a CPU this sum adds in a fixed order')

The two scatter-adds without the switch differ in their last bits, by a
few thousandths on bin sums of a few hundred; the deterministic computation
agrees with itself bitwise.

Bitwise agreement holds only for one machine, one library version, and one
configuration of these switches; no framework promises it across releases,
across hardware, or between CPU and GPU. Bitwise reproducibility is
therefore a tool for debugging: it lets us find the step at which two runs
first differ. The scientific goal is weaker and more robust. A conclusion,
such as "the change raises accuracy", should hold across seeds, so we report
the mean and the spread over several runs rather than one run, as for the
five seeds of that section (about 81%, with one to four
points between the best and the worst run). As with the
number formats of that section, a conclusion that changes with the
seed or with the last bits of the arithmetic needs more evidence.

## Model Inspection with Hooks

A repeatable run tells us that a problem will recur, not where it arises.
To find out, we observe what each layer computes during the run. Editing the
`forward` method of every layer to print or store its output is tedious, and
impossible for a model that came from a library or a checkpoint whose code
we do not want to change. A *hook* is a function that the framework calls at
a fixed point of each layer's computation, from outside the layer's code.

Hooks run in the `__call__` wrapper around `forward`
(that section). `module.register_forward_hook(f)`
arranges for `f(module, inputs, output)` to run after every forward pass of
the module, and `register_forward_pre_hook` before it;
the figure shows where they run. A forward hook that returns a
value replaces the module's output.

![The `__call__` wrapper as a pipeline: input flows through pre-hooks, then forward, then hooks, to the output, with the two hook stages dashed and orange against forward's solid blue, and a side arrow from the hooks stage to an observer that can capture, check, or modify.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-hooks.svg)

As a test model we use a stack of eight residual blocks of width 64 between
an input layer with 20 inputs and an output layer with 10 outputs. The block
is the one of that section: it adds the output of its
*body*, a linear layer, a ReLU, and a second linear layer, to its input,
$\mathbf{h} = \mathbf{x} + \textrm{body}(\mathbf{x})$. The weights start
from Xavier initialization (that section) and the biases at zero.
We do not seed this model. Its weights are drawn afresh in every run, which
is why the numbers below are quoted as ranges.

The seeded runs above left PyTorch's global generator at a fixed state,
which would fix this model's weights too, so the cell first reseeds the
generator from the operating system's entropy with `torch.seed()`.

In [ ]:
torch.seed()  # a fresh seed from the operating system

class ResidualBlock(nn.Module):
    def __init__(self, num_hiddens):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(num_hiddens, num_hiddens),
                                  nn.ReLU(),
                                  nn.Linear(num_hiddens, num_hiddens))

    def forward(self, X):
        return X + self.body(X)

net = nn.Sequential(nn.Linear(20, 64),
                    *[ResidualBlock(64) for _ in range(8)],
                    nn.Linear(64, 10))
for m in net.modules():
    if isinstance(m, nn.Linear):  # Xavier weights, zero biases
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

### Activation Statistics

The initialization experiments of that section measured the
standard deviation of the activations at every depth. With hooks, the same
measurement takes a few lines on an unmodified model. We record the standard
deviation of each layer's output on a minibatch of 256 standard normal
inputs.

In [ ]:
stats, handles = [], []

def record(module, inputs, output):
    stats.append((type(module).__name__, output.detach().std().item()))

for m in net:
    handles.append(m.register_forward_hook(record))
net(torch.randn(256, 20))
for h in handles:
    h.remove()
for name, std in stats:
    print(f'{name:15s} std {std:.2f}')

The standard deviation grows from block to block, from about 0.7 after the
input layer to roughly 3 to 4 after the eighth block. Both numbers can be
predicted. Xavier initialization gives the input layer, with 20 inputs and
64 outputs, weights of variance $2/84$, so its output has variance
$20 \cdot 2/84 \approx 0.48$ and standard deviation 0.69. Each block then
adds its body's output to its input, and the two are nearly uncorrelated, so
their variances add. The body's output has about half the variance of its
input: the first linear layer keeps the variance, the ReLU halves the mean
square, and the second linear layer keeps it again. The variance therefore
grows by a factor of about 1.5 per block, and the standard deviation by
about $\sqrt{1.5} \approx 1.22$. Eight blocks should multiply 0.69 by
$1.5^4 \approx 5$, which gives 3.5, and the table agrees to within the
scatter of one random draw. This is the calculation of
that section with a factor of 1.5 where He initialization gave
2, and it is the reason why the bodies of deep residual networks are often
initialized with smaller weights.

Two rules keep hooks safe. First, *detach before storing*: the hook above
stores `output.detach().std()`. Storing `output` itself would keep the
autograd graph of every forward pass in memory until `stats` is cleared, a
leak that grows with every minibatch. Second, *keep the handle and remove
it*: a hook stays registered for the module's lifetime unless we call
`handle.remove()`, slowing every later forward pass and keeping alive any
tensors it stores.

### A NaN Finder

When a loss becomes NaN at step 40,000, the useful question is which layer
produced the first non-finite value. A hook on every layer that checks its
output for infinities and NaNs answers it in one forward pass: the first
hook to fire names the layer, because hooks fire in the order the layers
compute. To test this, we put a NaN into one weight of the third residual
block, the module at index 3 of the stack:

In [ ]:
def make_finite_check(name):
    def check(module, inputs, output):
        if not torch.isfinite(output).all():
            raise RuntimeError(f'first non-finite output in {name}')
    return check

handles = [m.register_forward_hook(make_finite_check(name))
           for name, m in net.named_modules()
           if len(list(m.children())) == 0]  # leaf modules only
saved = net[3].body[0].weight[0, 0].item()
with torch.no_grad():
    net[3].body[0].weight[0, 0] = float('nan')  # break one layer
try:
    net(torch.randn(2, 20))
except RuntimeError as e:
    print(e)
for h in handles:
    h.remove()
with torch.no_grad():
    net[3].body[0].weight[0, 0] = saved  # restore the weight

The report names `3.body.0`, the layer that holds the NaN, without a single
print statement in the model.

A planted NaN is the easy case, since we knew where to look. In a real run
the finder names the first symptom, and the cause lies at or before it:
activations may grow over several layers before one of them overflows.

### Gradient Inspection

Gradients can be inspected in the same way. The forward signal grew with
depth. Does the backward signal do the same? We record the norm of the
gradient with respect to the output of each residual block, and the
gradient norms of the first linear layer of the third residual block. As a
loss we take the mean squared output of the stack on a minibatch of 256
standard normal inputs.

`module.register_full_backward_hook(f)` runs `f(module, grad_input,
grad_output)` during the backward pass, and `grad_output[0]` is the gradient
with respect to the module's output. Like a forward hook, a backward hook
returns a handle to remove and should detach anything it stores. The
gradients of the parameters need no hook: after the backward pass they are
available as `p.grad`.

In [ ]:
X = torch.randn(256, 20)
grad_norms, handles = [], []
def record(module, grad_input, grad_output):
    grad_norms.append(grad_output[0].norm().item())  # store a number
for block in net[1:-1]:
    handles.append(block.register_full_backward_hook(record))
net.zero_grad()
(net(X) ** 2).mean().backward()
for h in handles:
    h.remove()
# The hooks fire in backward order, last block first
print('block output gradients:',
      ' '.join(f'{g:.3g}' for g in reversed(grad_norms)))
layer = net[3].body[0]  # block 3, first layer
print(f'weight {layer.weight.grad.norm().item():.3g}, '
      f'bias {layer.bias.grad.norm().item():.3g}')

Read from the last block back to the first, the gradient norm grows by a
factor of roughly 1.2 to 1.4 per block, five- to sixfold over the stack.
This mirrors the forward growth of the activations, 1.22 per block, and has
the same cause: on the way back, too, each block adds the signal through
its body to the signal that bypasses it.

The two norms of a single layer mean little in isolation. They become
informative when compared across depth or across training steps, which the
exercises do.

What none of these tools can do is make two different machines agree, or
decide whether a difference between two runs matters; for that we compare
results across seeds. Nor did we carry out the whole procedure with which
the section opened, replaying a failed run from a checkpoint
(that section) with the finder attached, though all its parts
are now at hand.

The 8-billion-parameter model of the chapter introduction has not become
any smaller: its weights still take 32 GB in fp32. But we can now say how
such a model is built, initialized, stored, placed on a device, and
observed, each time by a walk over the same tree of modules. The chapters
that follow take these tools for granted, beginning with the convolutional
networks of that section.

## Summary

Every random draw in a training run comes from a generator fixed by a seed.
Seeding each generator the run uses fixes the draws, and private generator
objects or explicit keys keep unrelated draws from shifting them. Seeding
does not fix the arithmetic: floating-point addition depends on the order
of the terms, and some GPU operations add in a varying order, so bitwise
repetition also needs deterministic operations and holds only on one
machine and one software version. It is a tool for debugging; scientific
conclusions should hold across seeds. Hooks, or their equivalents in each
framework, observe the outputs and gradients of every layer of an unmodified
model, which turns the question of where the first NaN appeared into one
forward pass.

## Exercises

1. [code] **Block ablation.** A forward hook that returns a value
   *replaces* the module's output.
    1. Use such a hook to zero the output of one residual block's body,
       which turns the block into the identity, and measure the relative
       change in `net`'s output on a fixed batch. Repeat for every block,
       and rank the blocks by this change.
    1. Rebuild `net` three more times with new random weights and repeat
       the ranking. State whether the conclusion "block $k$ matters most"
       survives, and what evidence such a claim would need.
2. [code] **Residual variance budget.** Let $\mathbf{x}_k$,
   $\mathbf{b}_k$ and $\mathbf{h}_k$ denote the input, body output and
   output of block $k$, with each variance taken over all entries. On one
   batch, record for every `ResidualBlock` the variance of $\mathbf{x}_k$
   with `register_forward_pre_hook` (the pre-hook receives the argument
   tuple, so unpack it), of $\mathbf{h}_k$ with a forward hook on the
   block, and of $\mathbf{b}_k$ with a forward hook on its `body`. The
   standard deviations in the table of that section
   grow because each block adds its body's output to the stream. Measure
   how closely

   $$\textrm{Var}[\mathbf{h}_k] = \textrm{Var}[\mathbf{x}_k] + \textrm{Var}[\mathbf{b}_k]$$

   holds for every block, and name the term this equation omits.
3. [code] **Seeds and data loading.** ● Extend `train_once` to load its
   data through a `DataLoader` with `shuffle=True` and `num_workers=4`,
   keeping the single `torch.manual_seed(seed)` call at its top. The
   dataset adds noise to each example inside `__getitem__`, drawn from an
   `np.random.default_rng()` object created in the dataset's constructor.
   Run the code as a standalone script, because worker processes started
   with `spawn`, the default on macOS and Windows, cannot import a dataset
   class defined in a notebook.
    1. Predict whether two runs with the same seed agree on the shuffle
       order and on the noise, and whether the four workers draw the same
       noise. Justify each prediction from where that randomness is drawn,
       then verify all three.
    1. Add the `seed_worker` of this section and a seeded `generator` to
       the loader, and verify that two runs now agree and that the workers
       draw different noise. Does an example receive the same noise in
       the second epoch as in the first? Explain the answer from when the
       loader draws its base seed and calls the initializer.

    *Adapted from the PyTorch documentation
    [Multi-process data loading](https://docs.pytorch.org/docs/stable/data.html#randomness-in-multi-process-data-loading).*
4. [code] **Divergence monitor.** ● Turn the finiteness check of
   that section into a per-step training monitor. Rebuild
   `net` and train it with SGD at learning rate 1 on the mean squared
   error between its output and standard normal targets, using one fixed
   batch of 256 standard normal inputs. Report the first step at which a
   layer's output is non-finite, and the first such layer by depth. Check
   the loss at every step as well, and report whether it becomes
   non-finite before any layer output does.
5. [code] **Gradient norms.** ● Rebuild `net`, multiply the weight of
   every linear layer inside its residual blocks by 3, and use the mean
   squared output on a fixed batch as the loss.
    1. Record the norm of each block's output gradient as in this
       section. After the backward pass, compute each block's
       parameter-gradient norm, and plot both families against depth.
    1. Express the gradient of a linear layer's weight through the layer's
       input and its output gradient. Use this expression to explain why
       the two depth profiles differ, and state which family to monitor for
       an exploding backward signal.
6. [code] **Hook coverage.** Attach a forward hook to every leaf module of
   `torchvision.models.resnet18()` through `named_modules()`, and count
   how often each hook fires in one forward pass. Explain any module that
   fires more than once, and name a computation in `BasicBlock.forward`
   that no module hook observes.

[Discussions](https://d2l.discourse.group/t/36759)